# Data Loading and Preparation for O*NET Career Analysis

This notebook prepares O*NET occupational data for machine learning analysis. The goal is to combine **Abilities** and **Skills** datasets into a single, structured dataset that can be used for clustering jobs based on their required competencies.

## 1. Import Library

In [ ]:
# Data manipulation
import pandas as pd

## 2. Load the Data

We load two separate O*NET datasets:
- **Abilities**: Innate traits that influence how well someone can perform tasks (e.g., "Oral Comprehension", "Arm-Hand Steadiness")
- **Skills**: Learned competencies developed through training or experience (e.g., "Programming", "Critical Thinking")

In [ ]:
# Load Abilities dataset from O*NET
df_abilities = pd.read_excel('../Data/ONET/Abilities.xlsx')
print('Abilities')
print(df_abilities.head())

# Load Skills dataset from O*NET
df_skills = pd.read_excel('../Data/ONET/Skills.xlsx')
print('Skills')
print(df_skills.head())

## 3. Data Validation

Before merging datasets, we verify that both contain the same job titles.

In [ ]:
# Count unique jobs in each dataset
print("Unique job titles abilities: ", df_abilities['Title'].nunique())
print("Unique job titles skills: ", df_skills['Title'].nunique())

# Extract job title sets for comparison
ids_abilities = set(df_abilities['Title'].unique())
ids_skills = set(df_skills['Title'].unique())

# Verify both datasets contain the same jobs
print("Same job titles in both datasets: ", ids_abilities == ids_skills)

## 4. Filter for Level Values (LV)

O*NET provides two types of ratings for each skill/ability:
- **IM (Importance)**: How important is this skill for the job?
- **LV (Level)**: What level of proficiency is required?

In [ ]:
# Filter Abilities: keep only Level values ('LV'), not Importance ('IM')
df_abilities = df_abilities.loc[
    df_abilities['Scale ID'] == 'LV',
    ['O*NET-SOC Code', "Title", 'Element Name', 'Data Value']
]

df_abilities.head()

In [ ]:
# Filter Skills: same approach as Abilities
skills = df_skills.loc[
    df_skills['Scale ID'] == 'LV',
    ['O*NET-SOC Code', "Title", 'Element Name', 'Data Value']
]

skills.head()

## 5. Add Prefixes to Distinguish Features

Since both datasets have an "Element Name" column (the name of each skill or ability), we add prefixes:
- **"A: "** for Abilities
- **"S: "** for Skills

This prevents column name conflicts when merging and makes it easy to identify the source of each feature in the final dataset.

In [ ]:
df_abilities['Element Name'] = "A: " + df_abilities['Element Name']
df_skills['Element Name']    = "S: " + df_skills['Element Name']

## 6. Pivot Tables: Long to Wide Format

The original data is in "long format": one row per job-skill combination. For machine learning, we need "wide format": one row per job, with each skill/ability as its own column.

In [ ]:
# Pivot Skills: transform from long to wide format
skills_wide = df_skills.pivot_table(
    index='O*NET-SOC Code',      # Rows: unique job codes
    columns='Element Name',      # Columns: skill names
    values='Data Value'          # Cell values: proficiency levels
)

# Pivot Abilities: same transformation
# Include Title as index to preserve job names for later reference
abilities_wide = df_abilities.pivot_table(
    index=['O*NET-SOC Code', 'Title'],  # Multi-index to keep job title
    columns='Element Name',
    values='Data Value'
)

## 7. Merge Datasets

We join the Skills and Abilities datasets using the **O\*NET-SOC Code** (unique job identifier). This creates one comprehensive dataset with all features for each occupation.

**fillna(0)**: Any missing values are filled with 0, assuming that if a skill/ability isn't rated for a job, it's not required (level = 0).

In [ ]:
# Join Abilities and Skills into one dataset using SOC Code as the key
full_df = abilities_wide.join(skills_wide, on='O*NET-SOC Code')

# Handle missing values: fill with 0 (no skill required = level 0)
full_df = full_df.fillna(0)

# Verify the result
print("Shape:", full_df.shape) 
full_df.head()

## 8. Export Prepared Data

The final dataset is saved to CSV for use in the machine learning notebooks. This file contains:
- One row per occupation
- Columns for each Ability (prefixed "A:") and Skill (prefixed "S:")
- Values representing the required proficiency level (0-7 scale)

In [ ]:
# Save the prepared dataset for use in ML clustering notebooks
full_df.to_csv('../Data/career_pivot_results.csv')

### Output File
`career_pivot_results.csv` — Ready for use in clustering notebooks


### Next Steps
→ Proceed to K-means PCA notebook